# KW 19 Analyse: Inbound bis Fulfillment Verfolgung
## Produktweise Aggregation von Gewichten über Bereiche hinweg

Detaillierte Analyse der Transaction_Log.xlsx für Kalenderwoche 19:
- Verfolgung aller Produkte namentlich (ohne durchgängige Label-Nummern)
- Aggregation von Gewichten pro Produkt pro Bereich
- Bereichsübergreifende Verfolgung: Inbound → Zwischenlager → Fulfillment

## Sektion 1: Excel-Datei einlesen und laden

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Excel-Datei laden
file_path = r'c:\WMS Wahrheit\Transaction_Log.xlsx'
print(f"Lade Datei: {file_path}\n")

# Alle Sheets auflisten
xls = pd.ExcelFile(file_path)
print(f"Verfügbare Sheets: {xls.sheet_names}\n")

# Haupt-Sheet laden
df = pd.read_excel(file_path, sheet_name=0)

# Datenstruktur anzeigen
print(f"Datensatz-Abmessungen: {df.shape}")
print(f"\nSpaltennamen:\n{df.columns.tolist()}")
print(f"\nDatentypen:\n{df.dtypes}")
print(f"\nErste 5 Zeilen:")
print(df.head())

## Sektion 2: Daten für KW 19 filtern

In [ ]:
# Spalten für Datum/Woche identifizieren
print("Suche nach Datum- oder Wochenspalten...")

# Gängige Spaltennamen für Datum
date_cols = [col for col in df.columns if 'datum' in col.lower() or 'date' in col.lower() or 'zeit' in col.lower() or 'tag' in col.lower()]
week_cols = [col for col in df.columns if 'kw' in col.lower() or 'week' in col.lower() or 'woche' in col.lower() or 'calendar' in col.lower()]

print(f"Datums-Spalten gefunden: {date_cols}")
print(f"Wochenspalten gefunden: {week_cols}")

# Versuchen, KW 19 zu filtern
kw19_df = df.copy()

# Wenn eine Wochenspalte existiert
if week_cols:
    week_col = week_cols[0]
    kw19_df = df[df[week_col].astype(str).str.contains('19', case=False, na=False)]
    print(f"\nFilter nach Spalte '{week_col}' mit '19': {len(kw19_df)} Zeilen")
elif date_cols:
    date_col = date_cols[0]
    # KW 19 = 7. Mai bis 13. Mai 2026
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    kw19_df = df[(df[date_col] >= '2026-05-07') & (df[date_col] <= '2026-05-13')]
    print(f"\nFilter nach Datum (KW 19 = 7-13 Mai): {len(kw19_df)} Zeilen")
else:
    print("\nKeine Datums-/Wochenspalte gefunden. Zeige alle verfügbaren Werte:")
    for col in df.columns[:10]:
        print(f"\n{col}: {df[col].unique()[:5]}")

print(f"\nGefilterte KW 19 Datensätze: {kw19_df.shape}")
if len(kw19_df) > 0:
    print("\nErste 5 Zeilen KW 19:")
    print(kw19_df.head())

## Sektion 3: Produkte namentlich identifizieren und gruppieren

In [ ]:
# Produktnamen identifizieren
print("Suche nach Produktnamen-Spalten...")

product_cols = [col for col in kw19_df.columns if 'produkt' in col.lower() or 'product' in col.lower() or 'artikel' in col.lower() or 'sku' in col.lower() or 'name' in col.lower()]
print(f"Produktspalten gefunden: {product_cols}\n")

# Verfügbare Spalten mit 'Bereich' oder 'Lage' oder 'Zone'
location_cols = [col for col in kw19_df.columns if 'bereich' in col.lower() or 'lager' in col.lower() or 'zone' in col.lower() or 'location' in col.lower() or 'ort' in col.lower()]
print(f"Bereichs-/Standort-Spalten: {location_cols}\n")

# Gewichtsspalten
weight_cols = [col for col in kw19_df.columns if 'kg' in col.lower() or 'gewicht' in col.lower() or 'weight' in col.lower() or 'menge' in col.lower() or 'qty' in col.lower()]
print(f"Gewichts-/Mengenspalten: {weight_cols}\n")

# Alle restlichen Spalten
print(f"Alle Spalten insgesamt ({len(kw19_df.columns)}):")
for i, col in enumerate(kw19_df.columns):
    print(f"  {i}: {col}")
    
# Eindeutige Produktnamen (ohne Rack/Label-Nummern)
if product_cols:
    prod_col = product_cols[0]
    print(f"\n\nEindeutige Produkte in Spalte '{prod_col}':")
    unique_products = kw19_df[prod_col].unique()
    print(f"Anzahl einzigartiger Produkte: {len(unique_products)}")
    for p in unique_products[:20]:
        print(f"  - {p}")
    if len(unique_products) > 20:
        print(f"  ... und {len(unique_products) - 20} weitere")

## Sektion 4: Gewichte (kg) pro Produkt bereichsweise aggregieren

In [ ]:
# Bestimme relevante Spalten für Aggregation
prod_col = product_cols[0] if product_cols else None
location_col = location_cols[0] if location_cols else None
weight_col = weight_cols[0] if weight_cols else None

print(f"Verwendete Spalten:")
print(f"  Produkt: {prod_col}")
print(f"  Bereich: {location_col}")
print(f"  Gewicht: {weight_col}\n")

if prod_col and weight_col:
    # Konvertiere Gewichtsspalte zu numerisch
    kw19_df[weight_col] = pd.to_numeric(kw19_df[weight_col], errors='coerce')
    
    # Aggregation nach Produkt und Bereich
    if location_col:
        agg_by_area = kw19_df.groupby([prod_col, location_col])[weight_col].agg(['sum', 'count', 'mean']).reset_index()
        agg_by_area.columns = ['Produkt', 'Bereich', 'Gesamt_kg', 'Anzahl_Einträge', 'Durchschnitt_kg']
        print("AGGREGATION NACH PRODUKT UND BEREICH:")
        print(agg_by_area.to_string())
    
    # Aggregation nur nach Produkt
    agg_by_product = kw19_df.groupby(prod_col)[weight_col].agg(['sum', 'count', 'mean']).reset_index()
    agg_by_product.columns = ['Produkt', 'Gesamt_kg', 'Anzahl_Einträge', 'Durchschnitt_kg']
    print("\n\nAGGREGATION NACH PRODUKT (gesamt):")
    print(agg_by_product.to_string())
    
    # Speichere für nächste Sektionen
    area_summary = agg_by_area if location_col else None
    product_summary = agg_by_product
else:
    print("FEHLER: Produkt- oder Gewichtsspalte nicht gefunden!")

## Sektion 5: Inbound bis Fulfillment verfolgen

In [ ]:
# Identifiziere Bereiche/Prozessschritte
if location_col:
    bereiche = kw19_df[location_col].unique()
    bereiche_sorted = sorted(bereiche)
    
    print(f"Identifizierte Bereiche ({len(bereiche_sorted)}):")
    for i, bereich in enumerate(bereiche_sorted):
        print(f"  {i+1}. {bereich}")
    
    # Kategorisiere Bereiche in Prozessschritte
    print("\n\nBereichs-Klassifizierung:")
    inbound_keywords = ['inbound', 'empfang', 'wareneingang', 'receiving', 'wein']
    lager_keywords = ['lager', 'lagerbestand', 'storage', 'warehouse', 'zwischenlagr', 'zl']
    fulfillment_keywords = ['fulfillment', 'versand', 'shipping', 'fulfil', 'versandbereich', 'vb']
    
    inbound_areas = [b for b in bereiche_sorted if any(k in str(b).lower() for k in inbound_keywords)]
    lager_areas = [b for b in bereiche_sorted if any(k in str(b).lower() for k in lager_keywords)]
    fulfillment_areas = [b for b in bereiche_sorted if any(k in str(b).lower() for k in fulfillment_keywords)]
    other_areas = [b for b in bereiche_sorted if b not in inbound_areas + lager_areas + fulfillment_areas]
    
    print(f"\nInbound: {inbound_areas}")
    print(f"Lager/Zwischenlager: {lager_areas}")
    print(f"Fulfillment/Versand: {fulfillment_areas}")
    print(f"Andere: {other_areas}")
    
    # Verfolge Produkt-Flow pro Bereich
    print("\n\nPRODUKT-FLOW VON INBOUND BIS FULFILLMENT:")
    print("="*80)
    
    unique_products_list = kw19_df[prod_col].unique()
    
    flow_data = []
    for product in unique_products_list:
        prod_data = kw19_df[kw19_df[prod_col] == product]
        
        inbound_qty = prod_data[prod_data[location_col].isin(inbound_areas)][weight_col].sum()
        lager_qty = prod_data[prod_data[location_col].isin(lager_areas)][weight_col].sum()
        fulfillment_qty = prod_data[prod_data[location_col].isin(fulfillment_areas)][weight_col].sum()
        other_qty = prod_data[prod_data[location_col].isin(other_areas)][weight_col].sum()
        
        flow_data.append({
            'Produkt': product,
            'Inbound_kg': inbound_qty if inbound_qty > 0 else 0,
            'Lager_kg': lager_qty if lager_qty > 0 else 0,
            'Fulfillment_kg': fulfillment_qty if fulfillment_qty > 0 else 0,
            'Andere_kg': other_qty if other_qty > 0 else 0,
            'Gesamt_kg': inbound_qty + lager_qty + fulfillment_qty + other_qty
        })
    
    flow_df = pd.DataFrame(flow_data)
    print(flow_df.to_string())
else:
    print("Keine Bereichs-/Ortsspalte gefunden. Flow-Verfolgung nicht möglich.")

## Sektion 6: Bereichsübergreifende Gewichte zusammenfassen

In [ ]:
# Zusammenfassung der Gesamtgewichte
print("ÜBERSICHTSTABELLE: GESAMTGEWICHTE PRO PRODUKT")
print("="*80)

summary_all = kw19_df.groupby(prod_col)[weight_col].agg(['sum', 'count', 'min', 'max', 'mean']).reset_index()
summary_all.columns = ['Produkt', 'Gesamt_kg', 'Anzahl_Transaktionen', 'Min_kg_pro_Eintrag', 'Max_kg_pro_Eintrag', 'Durchschnitt_kg']
summary_all = summary_all.sort_values('Gesamt_kg', ascending=False)

print(summary_all.to_string())

print(f"\n\nGESAMTSUMME KW 19: {summary_all['Gesamt_kg'].sum():.2f} kg")
print(f"Anzahl einzigartiger Produkte: {len(summary_all)}")
print(f"Anzahl Transaktionen: {summary_all['Anzahl_Transaktionen'].sum():.0f}")

# Top 10 Produkte nach Gewicht
print("\n\nTOP 10 PRODUKTE NACH GEWICHT:")
print("-"*80)
top10 = summary_all.head(10)
print(top10.to_string())

# Gewichtsverteilung nach Bereichen (Gesamt)
if location_col:
    print("\n\nGEWICHTSVERTEILUNG NACH BEREICHEN:")
    print("-"*80)
    area_totals = kw19_df.groupby(location_col)[weight_col].sum().sort_values(ascending=False)
    for area, weight in area_totals.items():
        pct = (weight / area_totals.sum()) * 100
        print(f"{area:30s}: {weight:10.2f} kg ({pct:5.1f}%)")

## Sektion 7: Ergebnisse visualisieren und exportieren

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisierungen
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Top 10 Produkte nach Gewicht (Balkendiagramm)
top10 = summary_all.head(10)
ax1 = axes[0, 0]
ax1.barh(range(len(top10)), top10['Gesamt_kg'].values)
ax1.set_yticks(range(len(top10)))
ax1.set_yticklabels(top10['Produkt'].values, fontsize=9)
ax1.set_xlabel('Gewicht (kg)')
ax1.set_title('Top 10 Produkte nach Gewicht (KW 19)')
ax1.invert_yaxis()

# 2. Gewichtsverteilung nach Bereich (Kuchen)
if location_col:
    ax2 = axes[0, 1]
    area_totals = kw19_df.groupby(location_col)[weight_col].sum()
    ax2.pie(area_totals.values, labels=area_totals.index, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Gewichtsverteilung nach Bereich')

# 3. Produktfluss Inbound -> Fulfillment
if 'flow_df' in locals():
    ax3 = axes[1, 0]
    flow_data_plot = flow_df[['Inbound_kg', 'Lager_kg', 'Fulfillment_kg']].sum()
    ax3.bar(flow_data_plot.index, flow_data_plot.values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax3.set_ylabel('Gewicht (kg)')
    ax3.set_title('Gesamt-Materialfluss: Inbound → Lager → Fulfillment')
    ax3.tick_params(axis='x', rotation=45)

# 4. Anzahl Transaktionen pro Produkt (Top 10)
ax4 = axes[1, 1]
top10_trans = summary_all.nlargest(10, 'Anzahl_Transaktionen')
ax4.bar(range(len(top10_trans)), top10_trans['Anzahl_Transaktionen'].values, color='steelblue')
ax4.set_xticks(range(len(top10_trans)))
ax4.set_xticklabels(top10_trans['Produkt'].values, rotation=45, ha='right', fontsize=8)
ax4.set_ylabel('Anzahl Transaktionen')
ax4.set_title('Top 10 Produkte nach Transaktionsanzahl')

plt.tight_layout()
plt.savefig(r'c:\WMS Wahrheit\KW19_Analyse_Visualisierung.png', dpi=150, bbox_inches='tight')
print("Visualisierung gespeichert: KW19_Analyse_Visualisierung.png")
plt.show()

# Exportiere Ergebnisse in XLSX
output_file = r'c:\WMS Wahrheit\KW19_Analyse_Ergebnisse.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Zusammenfassung nach Produkt
    summary_all.to_excel(writer, sheet_name='Produkte_Gesamt', index=False)
    
    # Sheet 2: Flow-Daten
    if 'flow_df' in locals():
        flow_df.to_excel(writer, sheet_name='Produkt_Flow', index=False)
    
    # Sheet 3: Bereichs-Aggregation
    if 'area_summary' in locals():
        area_summary.to_excel(writer, sheet_name='Nach_Bereich', index=False)
    
    # Sheet 4: Rohdaten KW 19
    kw19_df.to_excel(writer, sheet_name='Rohdaten_KW19', index=False)

print(f"\n✓ Ergebnisse exportiert: {output_file}")
print(f"\nExportierte Sheets:")
print("  1. Produkte_Gesamt - Zusammenfassung aller Produkte")
print("  2. Produkt_Flow - Materialfluss von Inbound bis Fulfillment")
if 'area_summary' in locals():
    print("  3. Nach_Bereich - Aggregation nach Produkt und Bereich")
print(f"  4. Rohdaten_KW19 - Originaldaten für KW 19")